# Multimodal Agent Patterns

This notebook collects patterns for agents that work across text, images, OCR output, and audio transcripts.

The goal is not to cram every modality into one prompt. The goal is to normalize each input type and then let the agent reason over a common representation.


## The multimodal pipeline

A practical pipeline usually looks like this:

1. detect modality
2. extract text or signals
3. normalize into a shared schema
4. retrieve supporting evidence
5. reason with the right model
6. cite the evidence and hand off low-confidence cases


In [ ]:
from dataclasses import dataclass

@dataclass
class MultimodalInput:
    modality: str
    source: str
    extracted_text: str
    confidence: float

doc = MultimodalInput(
    modality='image',
    source='invoice_scan.png',
    extracted_text='Invoice total $1840.55 due in 30 days',
    confidence=0.88,
)

doc


In [ ]:
def normalize_multimodal_input(item: MultimodalInput) -> dict[str, object]:
    return {
        'modality': item.modality,
        'source': item.source,
        'text': item.extracted_text.strip(),
        'confidence': item.confidence,
        'needs_review': item.confidence < 0.9,
    }

normalize_multimodal_input(doc)


## Model routing tips

- use a small text model when the document is already clean text
- use a vision-language model for screenshots, forms, and tables
- use OCR before the LLM when the source is scanned
- keep a GPU-backed path available for heavier workloads
- if you are serving a local model, a vLLM endpoint can replace the inference layer later


In [ ]:
def route_multimodal_model(item: MultimodalInput) -> str:
    if item.modality in {'image', 'pdf'}:
        return 'vision_language_model'
    if item.modality == 'audio':
        return 'speech_to_text_then_reason'
    return 'text_model'

route_multimodal_model(doc)


## Useful patterns for real systems

- keep raw artifacts for auditability
- store extracted text separately from the final answer
- attach confidence and source location
- support human review when the extraction is uncertain
- do not let a vision model override business validation rules


## Practical examples

This pattern shows up in:

- invoice extraction
- claims processing
- onboarding document checks
- screenshot-based QA
- field service notes
- voice call summaries with supporting evidence


## Demo ideas

Great demo inputs for this notebook:

- scanned PDF page
- chart screenshot
- photo of a form
- meeting transcript
- labeled table image
